# Bridgr ML Model - Google Colab Ready Notebook

This notebook contains the complete ML model from the Bridgr backend, ready to run in Google Colab.

## Features:
- Resume parsing and skill extraction
- Job matching and similarity analysis
- Skill gap analysis with learning recommendations
- Transferable skill detection
- Salary estimation for Indian market

## Setup Instructions:
1. Run all cells in order
2. Upload your resume PDF when prompted
3. Enter target job role
4. Get comprehensive analysis results

## 1. Install Dependencies

In [ ]:
# Install required packages
!pip install pdfplumber spacy sentence-transformers scikit-learn pandas numpy python-dotenv openai pydantic

# Download spaCy model
!python -m spacy download en_core_web_sm

print("All dependencies installed successfully!")

## 2. Import Dependencies and Setup

In [ ]:
import os
import re
import json
import hashlib
import zipfile
import glob
import numpy as np
import pandas as pd
import pdfplumber
from pathlib import Path
from typing import List, Dict, Set, Tuple, Optional, Union
from collections import Counter, defaultdict
from pydantic import BaseModel
import spacy
from spacy.matcher import PhraseMatcher
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# For file uploads
from google.colab import files
import io

print("All imports successful!")

## 3. Data Models (Pydantic)

In [ ]:
class ExtractedSkill(BaseModel):
    """One skill found in the user's resume."""
    name: str
    normalized: str
    confidence: float
    source: str = "resume"
    context: str = ""

class SkillGap(BaseModel):
    """A skill the user is missing, with context about how important it is."""
    name: str
    priority: str
    priority_score: float = 0.5
    market_demand: float = 0.5
    reason: str
    estimated_weeks: int = 4
    has_foundation: bool = False
    demand_percentage: int = 50
    learning_resources: List[str] = []

class TransferableSkill(BaseModel):
    """A skill the user has that partially satisfies a requirement they're missing."""
    user_skill: str
    maps_to_job_skill: str
    transfer_score: float
    explanation: str

class AnalysisResult(BaseModel):
    """The complete output of one resume analysis."""
    analysis_id: str
    generated_at: str
    target_role: str
    match_score: int
    readiness_level: str
    confidence_score: float
    extracted_skills: List[ExtractedSkill]
    matched_skills: List[str]
    missing_required: List[SkillGap]
    missing_preferred: List[SkillGap]
    transferable_skills: List[TransferableSkill]
    priority_skills: List[str]
    market_demand_skills: List[str]
    learning_roadmap_inputs: Dict
    mock_interview_inputs: Dict
    career_chat_context: Dict
    salary_band_estimate: Dict
    explanations: List[str]

print("Data models defined!")

## 4. Fallback Skills Database

In [ ]:
print("Fallback skills database loaded!")

## 5. Resume Parser

In [ ]:
print("Resume parser class defined!")

## 6. Skill Extractor

In [ ]:
print("Loading NLP models...")
        self.nlp = spacy.load("en_core_web_sm")
        self.embed_model = SentenceTransformer("all-MiniLM-L6-v2")

        # Build PhraseMatcher
        self._matcher = PhraseMatcher(self.nlp.vocab, attr="LOWER")
        patterns = list(self.nlp.pipe(self.skill_list))
        self._matcher.add("SKILLS", patterns)

        # Precompute all skill embeddings
        print(f"Precomputing embeddings for {len(self.skill_list)} skills...")
        self._skill_embeddings = self.embed_model.encode(
            self.skill_list,
            batch_size=64,
            normalize_embeddings=True,
            show_progress_bar=True,
        )
        print("Skill extractor ready")

## 7. Similarity Engine

In [ ]:
class MatchingEngine:
    """Computes a semantic + lexical hybrid match score between user skills and job requirements."""

    def __init__(self, embed_model: SentenceTransformer):
        self.embed_model = embed_model
        self._vec_cache: Dict[str, np.ndarray] = {}

    def compute_match(self, user_skills: List[str], job_tech_skills: List[str], job_soft_skills: List[str]) -> Tuple[int, float]:
        """Returns: (match_score 0–100, confidence 0–1)"""
        if not user_skills or not job_tech_skills:
            return 0, 0.5

        user_set = set(s.lower().strip() for s in user_skills)
        job_tech = set(s.lower().strip() for s in job_tech_skills)
        job_soft = set(s.lower().strip() for s in job_soft_skills)
        job_all = job_tech | job_soft

        if not job_all:
            return 0, 0.3

        # Component 1: Weighted Jaccard
        tech_matched = len(user_set & job_tech)
        soft_matched = len(user_set & job_soft)

        numerator = tech_matched * 3.0 + soft_matched * 1.0
        denominator = len(job_tech) * 3.0 + len(job_soft) * 1.0
        weighted_jaccard = numerator / denominator if denominator > 0 else 0.0

        # Component 2: Semantic similarity of entire skill sets
        user_vec = self._embed_skill_set(list(user_set))
        job_vec = self._embed_skill_set(list(job_all))
        semantic_sim = float(cosine_similarity([user_vec], [job_vec])[0][0])
        semantic_sim = max(0.0, semantic_sim)

        # Blend
        raw = 0.60 * semantic_sim + 0.40 * weighted_jaccard

        # Scale: 0.5 raw → 65 displayed (50% skill overlap is actually decent)
        scaled = min(100, int(raw * 100 * 1.15))

        # Confidence: higher when both sides have more skills to compare
        coverage = min(len(user_set), len(job_all)) / max(len(job_all), 1)
        confidence = round(min(0.95, 0.5 + coverage * 0.45), 2)

        return scaled, confidence

    def find_transferable_skills(self, user_skills: List[str], missing_skills: List[str], min_score: float = 0.72) -> List[TransferableSkill]:
        """For each missing skill, check if the user has something similar."""
        if not user_skills or not missing_skills:
            return []

        user_vecs = self.embed_model.encode(user_skills, normalize_embeddings=True)
        missing_vecs = self.embed_model.encode(missing_skills, normalize_embeddings=True)

        results = []
        for missing_skill, missing_vec in zip(missing_skills, missing_vecs):
            sims = np.dot(user_vecs, missing_vec)
            best_idx = int(np.argmax(sims))
            best_sim = float(sims[best_idx])

            if best_sim >= min_score:
                user_skill = user_skills[best_idx]
                results.append(TransferableSkill(
                    user_skill=user_skill,
                    maps_to_job_skill=missing_skill,
                    transfer_score=round(best_sim, 3),
                    explanation=(
                        f"Your '{user_skill}' experience shows {int(best_sim*100)}% "
                        f"overlap with '{missing_skill}'. "
                        f"You'll need less time to learn this than a complete beginner."
                    )
                ))

        return results

    def _embed_skill_set(self, skills: List[str]) -> np.ndarray:
        """Embed a list of skills into one vector by mean-pooling."""
        if not skills:
            return np.zeros(384)   # all-MiniLM-L6-v2 output dim

        # Cache by sorted skill list (same skills in different order = same vector)
        cache_key = hashlib.md5("|".join(sorted(skills)).encode()).hexdigest()
        if cache_key in self._vec_cache:
            return self._vec_cache[cache_key]

        vecs = self.embed_model.encode(skills, normalize_embeddings=True)
        pooled = np.mean(vecs, axis=0)

        # Re-normalize after pooling
        norm = np.linalg.norm(pooled)
        if norm > 0:
            pooled = pooled / norm

        self._vec_cache[cache_key] = pooled
        return pooled

print("✅ Matching engine class defined!")

## 8. Gap Analyzer

In [ ]:
# Prerequisite relationships
PREREQUISITE_MAP = {
    "machine learning": ["python", "statistics", "linear algebra", "numpy"],
    "deep learning": ["machine learning", "python", "tensorflow", "pytorch"],
    "natural language processing": ["python", "machine learning", "text mining"],
    "data science": ["python", "statistics", "sql", "machine learning"],
    "feature engineering": ["python", "pandas", "statistics"],
    "model deployment": ["python", "docker", "flask", "fastapi"],
    "mlops": ["docker", "kubernetes", "python", "ci/cd"],
    "sql": [],
    "pandas": ["python"],
    "tensorflow": ["python", "machine learning", "numpy"],
    "pytorch": ["python", "machine learning", "numpy"],
    "spark": ["python", "sql", "hadoop"],
    "kubernetes": ["docker", "linux"],
    "react": ["javascript", "html", "css"],
    "node.js": ["javascript"],
    "django": ["python"],
    "fastapi": ["python"],
    "aws": ["linux", "networking"],
    "gcp": ["linux"],
    "azure": ["linux"],
    "data visualization": ["python", "matplotlib"],
    "tableau": [],
    "power bi": [],
    "a/b testing": ["statistics"],
    "time series": ["statistics", "python", "pandas"],
}

# Estimated weeks for someone with some technical background
LEARNING_TIME_MAP = {
    "sql": 3, "python": 6, "pandas": 2, "numpy": 2,
    "machine learning": 8, "deep learning": 10, "tensorflow": 4, "pytorch": 4,
    "spark": 6, "docker": 3, "kubernetes": 6, "react": 8, "javascript": 8,
    "aws": 8, "statistics": 6, "feature engineering": 4, "data visualization": 3,
    "tableau": 2, "nlp": 6, "mlops": 8,
    "default": 4,
}

# Indian salary bands by role keyword
INDIA_SALARY_BANDS = {
    "data scientist": {"min": 700000, "max": 2000000, "median": 1200000, "currency": "INR"},
    "data analyst": {"min": 400000, "max": 1200000, "median": 700000, "currency": "INR"},
    "machine learning": {"min": 900000, "max": 2500000, "median": 1600000, "currency": "INR"},
    "software engineer": {"min": 500000, "max": 2000000, "median": 1000000, "currency": "INR"},
    "frontend": {"min": 400000, "max": 1500000, "median": 800000, "currency": "INR"},
    "backend": {"min": 500000, "max": 1800000, "median": 1000000, "currency": "INR"},
    "fullstack": {"min": 500000, "max": 2000000, "median": 1100000, "currency": "INR"},
    "product manager": {"min": 800000, "max": 2500000, "median": 1500000, "currency": "INR"},
    "devops": {"min": 600000, "max": 2000000, "median": 1200000, "currency": "INR"},
    "default": {"min": 400000, "max": 1500000, "median": 800000, "currency": "INR"},
}

HIGH_DEMAND_THRESHOLD = 0.15

class GapAnalyzer:
    """Computes prioritized skill gaps between a user and a job profile."""

    def __init__(self, skill_market_demand: Dict[str, float]):
        self.skill_market_demand = skill_market_demand

    def analyze(self, user_skills: List[str], job_tech_skills: List[str], job_soft_skills: List[str], transferable: List[TransferableSkill]) -> Tuple[List[SkillGap], List[SkillGap]]:
        """Returns: (missing_required_ranked, missing_preferred_ranked)"""
        user_set = set(s.lower().strip() for s in user_skills)
        job_tech = set(s.lower().strip() for s in job_tech_skills)
        job_soft = set(s.lower().strip() for s in job_soft_skills)
        transferable_targets = {t.maps_to_job_skill.lower() for t in transferable}

        missing_tech = job_tech - user_set
        missing_soft = job_soft - user_set

        required_gaps = [self._build_gap(s, user_set, transferable_targets, True) for s in missing_tech]
        preferred_gaps = [self._build_gap(s, user_set, transferable_targets, False) for s in missing_soft]

        required_gaps = sorted(required_gaps, key=lambda g: g.priority_score, reverse=True)
        preferred_gaps = sorted(preferred_gaps, key=lambda g: g.priority_score, reverse=True)

        return required_gaps, preferred_gaps

    def _build_gap(self, skill: str, user_set: Set[str], transferable_targets: Set[str], is_required: bool) -> SkillGap:
        market_demand = self.skill_market_demand.get(skill, 0.05)
        weeks = LEARNING_TIME_MAP.get(skill, LEARNING_TIME_MAP["default"])
        max_weeks = max(LEARNING_TIME_MAP.values())
        difficulty = weeks / max_weeks

        prereqs = PREREQUISITE_MAP.get(skill, [])
        prereqs_met = sum(1 for p in prereqs if p in user_set)
        has_foundation = prereqs_met > 0 and len(prereqs) > 0
        foundation_bonus = prereqs_met / len(prereqs) if prereqs else 0.0

        transfer_bonus = 0.5 if skill in transferable_targets else 0.0

        priority_score = round(
            0.40 * market_demand
            + 0.30 * (1 - difficulty)
            + 0.20 * foundation_bonus
            + 0.10 * transfer_bonus,
            4
        )

        if priority_score >= 0.35:   priority = "Critical"
        elif priority_score >= 0.25: priority = "High"
        elif priority_score >= 0.15: priority = "Medium"
        else:                        priority = "Low"

        # Build human reason
        parts = []
        if market_demand > HIGH_DEMAND_THRESHOLD:
            parts.append(f"required by {int(market_demand*100)}% of job postings")
        if has_foundation:
            met = [p for p in prereqs if p in user_set]
            parts.append(f"you already know {', '.join(met[:2])} (related)")
        if skill in transferable_targets:
            parts.append("you have a transferable skill that covers this partially")
        reason = "; ".join(parts) if parts else f"important for {skill} roles"

        return SkillGap(
            name=skill,
            priority=priority,
            priority_score=priority_score,
            market_demand=market_demand,
            reason=reason,
            estimated_weeks=weeks,
            has_foundation=has_foundation,
        )

    def get_salary_band(self, target_role: str) -> Dict:
        role_lower = target_role.lower()
        for keyword, band in INDIA_SALARY_BANDS.items():
            if keyword in role_lower:
                return band
        return INDIA_SALARY_BANDS["default"]

print("✅ Gap analyzer class defined!")

## 9. Main Analysis Pipeline

In [ ]:
import uuid
from datetime import datetime

class BridgrAnalyzer:
    """Main analyzer that combines all components."""

    def __init__(self, openai_api_key: str = ""):
        self.openai_key = openai_api_key
        self.resume_parser = ResumeParser()
        
        # Build skill list from fallback database
        all_skills = set()
        for role_data in FALLBACK_JOB_SKILLS.values():
            all_skills.update(role_data["tech_skills"])
            all_skills.update(role_data["soft_skills"])
        
        self.skill_extractor = SkillExtractor(
            skill_list=list(all_skills),
            semantic_threshold=0.75,
            openai_key=openai_api_key
        )
        
        self.matching_engine = MatchingEngine(self.skill_extractor.embed_model)
        
        # Create mock market demand data
        self.skill_market_demand = self._create_mock_market_demand()
        self.gap_analyzer = GapAnalyzer(self.skill_market_demand)

    def _create_mock_market_demand(self) -> Dict[str, float]:
        """Create mock market demand data based on skill importance."""
        demand = {}
        high_demand_skills = [
            "python", "sql", "machine learning", "aws", "docker", 
            "javascript", "react", "communication", "problem solving"
        ]
        medium_demand_skills = [
            "pandas", "numpy", "tensorflow", "pytorch", "git",
            "data analysis", "statistics", "teamwork", "project management"
        ]
        
        for skill in high_demand_skills:
            demand[skill] = 0.25
        for skill in medium_demand_skills:
            demand[skill] = 0.15
        
        # Add all other skills with low demand
        all_skills = set()
        for role_data in FALLBACK_JOB_SKILLS.values():
            all_skills.update(role_data["tech_skills"])
            all_skills.update(role_data["soft_skills"])
        
        for skill in all_skills:
            if skill not in demand:
                demand[skill] = 0.08
        
        return demand

    def analyze_resume(self, pdf_path: str, target_role: str) -> AnalysisResult:
        """Main analysis pipeline."""
        print(f"🔍 Analyzing resume for {target_role}...")
        
        # Step 1: Parse resume
        resume_data = self.resume_parser.parse(pdf_path)
        print(f"📄 Resume parsed: {resume_data['metadata']['char_count']} characters")
        
        # Step 2: Extract skills
        extracted_skills = self.skill_extractor.extract(resume_data, debug=True)
        user_skills = [s.normalized for s in extracted_skills]
        print(f"🎯 Skills extracted: {len(extracted_skills)}")
        
        # Step 3: Get job requirements
        job_profile = get_job_skills(target_role)
        job_tech_skills = job_profile["tech_skills"]
        job_soft_skills = job_profile["soft_skills"]
        print(f"💼 Job profile loaded: {len(job_tech_skills)} tech, {len(job_soft_skills)} soft skills")
        
        # Step 4: Compute match score
        match_score, confidence = self.matching_engine.compute_match(
            user_skills, job_tech_skills, job_soft_skills
        )
        print(f"📊 Match score: {match_score}% (confidence: {confidence})")
        
        # Step 5: Find transferable skills
        all_job_skills = job_tech_skills + job_soft_skills
        missing_skills = [s for s in all_job_skills if s.lower() not in user_skills]
        transferable_skills = self.matching_engine.find_transferable_skills(
            user_skills, missing_skills
        )
        print(f"🔄 Transferable skills found: {len(transferable_skills)}")
        
        # Step 6: Analyze skill gaps
        missing_required, missing_preferred = self.gap_analyzer.analyze(
            user_skills, job_tech_skills, job_soft_skills, transferable_skills
        )
        print(f"📈 Skill gaps: {len(missing_required)} required, {len(missing_preferred)} preferred")
        
        # Step 7: Determine readiness level
        if match_score >= 85: readiness = "Ready"
        elif match_score >= 70: readiness = "Almost Ready"
        elif match_score >= 55: readiness = "Getting There"
        else: readiness = "Needs Work"
        
        # Step 8: Get matched skills
        user_set = set(user_skills)
        job_tech_set = set(s.lower() for s in job_tech_skills)
        job_soft_set = set(s.lower() for s in job_soft_skills)
        matched_skills = list(user_set & (job_tech_set | job_soft_set))
        
        # Step 9: Get priority skills (top 5 missing skills)
        priority_skills = [gap.name for gap in (missing_required + missing_preferred)][:5]
        
        # Step 10: Get market demand skills
        market_demand_skills = sorted(
            [skill for skill, demand in self.skill_market_demand.items() if demand > 0.15],
            key=lambda x: self.skill_market_demand[x],
            reverse=True
        )[:10]
        
        # Step 11: Get salary estimate
        salary_band = self.gap_analyzer.get_salary_band(target_role)
        
        # Step 12: Build explanations
        explanations = [
            f"Your profile matches {match_score}% with {target_role} requirements.",
            f"You have {len(matched_skills)} of the required skills.",
            f"Focus on learning: {', '.join(priority_skills[:3])} to improve your match."
        ]
        
        # Step 13: Build context for downstream features
        learning_roadmap_inputs = {
            "missing_required": [gap.dict() for gap in missing_required],
            "missing_preferred": [gap.dict() for gap in missing_preferred],
            "transferable_skills": [ts.dict() for ts in transferable_skills],
            "user_skills": user_skills,
            "target_role": target_role
        }
        
        mock_interview_inputs = {
            "target_role": target_role,
            "user_skills": user_skills,
            "missing_skills": priority_skills,
            "matched_skills": matched_skills
        }
        
        career_chat_context = {
            "target_role": target_role,
            "match_score": match_score,
            "user_skills": user_skills,
            "skill_gaps": priority_skills,
            "readiness_level": readiness
        }
        
        # Build final result
        result = AnalysisResult(
            analysis_id=str(uuid.uuid4()),
            generated_at=datetime.now().isoformat(),
            target_role=target_role,
            match_score=match_score,
            readiness_level=readiness,
            confidence_score=confidence,
            extracted_skills=extracted_skills,
            matched_skills=matched_skills,
            missing_required=missing_required,
            missing_preferred=missing_preferred,
            transferable_skills=transferable_skills,
            priority_skills=priority_skills,
            market_demand_skills=market_demand_skills,
            learning_roadmap_inputs=learning_roadmap_inputs,
            mock_interview_inputs=mock_interview_inputs,
            career_chat_context=career_chat_context,
            salary_band_estimate=salary_band,
            explanations=explanations
        )
        
        print("✅ Analysis complete!")
        return result

print("✅ Main analyzer class defined!")

## 10. Upload and Analyze Your Resume

In [ ]:
# Optional: Set your OpenAI API key for enhanced skill extraction
# Leave empty to use only the built-in extraction methods
OPENAI_API_KEY = ""  # @param {type:"string"}

# Initialize the analyzer
print("🚀 Initializing Bridgr Analyzer...")
analyzer = BridgrAnalyzer(openai_api_key=OPENAI_API_KEY)
print("✅ Analyzer ready!")

In [ ]:
# Upload your resume PDF
print("📤 Please upload your resume PDF file:")
uploaded = files.upload()

if not uploaded:
    print("❌ No file uploaded. Please run this cell again and upload a PDF.")
else:
    pdf_filename = list(uploaded.keys())[0]
    print(f"✅ File uploaded: {pdf_filename}")
    
    # Save the uploaded file
    with open(pdf_filename, 'wb') as f:
        f.write(uploaded[pdf_filename])
    
    print(f"💾 File saved as: {pdf_filename}")

In [ ]:
# Enter your target job role
target_role = input("🎯 Enter your target job role (e.g., 'Data Scientist', 'Software Engineer'): ").strip()

if not target_role:
    print("❌ Please enter a target job role.")
else:
    print(f"🎯 Analyzing for role: {target_role}")

In [ ]:
# Run the complete analysis
if 'pdf_filename' in locals() and 'target_role' in locals() and target_role:
    try:
        result = analyzer.analyze_resume(pdf_filename, target_role)
        
        # Store result for later use
        analysis_result = result
        
        print("\n" + "="*80)
        print("🎉 ANALYSIS RESULTS")
        print("="*80)
        
    except Exception as e:
        print(f"❌ Error during analysis: {e}")
        print("Please check that your PDF is text-based and try again.")
else:
    print("❌ Please upload a resume and enter a target role first.")

## 11. Display Results

In [ ]:
# Display comprehensive results
if 'analysis_result' in locals():
    result = analysis_result
    
    print(f"\n🎯 TARGET ROLE: {result.target_role.upper()}")
    print(f"📊 MATCH SCORE: {result.match_score}%")
    print(f"📈 READINESS LEVEL: {result.readiness_level}")
    print(f"🔮 CONFIDENCE: {result.confidence_score:.2f}")
    
    # Salary information
    salary = result.salary_band_estimate
    print(f"\n💰 SALARY ESTIMATE (India):")
    print(f"   Range: {salary['currency']} {salary['min']:,} - {salary['max']:,}")
    print(f"   Median: {salary['currency']} {salary['median']:,}")
    
    # Skills summary
    print(f"\n🎯 SKILLS SUMMARY:")
    print(f"   ✅ Matched Skills: {len(result.matched_skills)}")
    print(f"   ❌ Missing Required: {len(result.missing_required)}")
    print(f"   💡 Missing Preferred: {len(result.missing_preferred)}")
    print(f"   🔄 Transferable Skills: {len(result.transferable_skills)}")
    
    # Matched skills
    if result.matched_skills:
        print(f"\n✅ MATCHED SKILLS:")
        for skill in result.matched_skills[:10]:
            print(f"   • {skill}")
        if len(result.matched_skills) > 10:
            print(f"   ... and {len(result.matched_skills) - 10} more")
    
    # Priority skills to learn
    if result.priority_skills:
        print(f"\n🔥 PRIORITY SKILLS TO LEARN:")
        for i, skill in enumerate(result.priority_skills[:5], 1):
            print(f"   {i}. {skill}")
    
    # Top missing required skills
    if result.missing_required:
        print(f"\n❌ TOP MISSING REQUIRED SKILLS:")
        for gap in result.missing_required[:5]:
            print(f"   • {gap.name} ({gap.priority}) - {gap.reason}")
    
    # Transferable skills
    if result.transferable_skills:
        print(f"\n🔄 TRANSFERABLE SKILLS:")
        for ts in result.transferable_skills[:5]:
            print(f"   • '{ts.user_skill}' → '{ts.maps_to_job_skill}' ({int(ts.transfer_score*100)}% overlap)")
    
    # Explanations
    if result.explanations:
        print(f"\n💡 KEY INSIGHTS:")
        for explanation in result.explanations:
            print(f"   • {explanation}")
    
    print("\n" + "="*80)
    
else:
    print("❌ No analysis results available. Please run the analysis first.")

## 12. Detailed Skill Breakdown

In [ ]:
# Show detailed extracted skills
if 'analysis_result' in locals():
    result = analysis_result
    
    print("🔍 DETAILED SKILL EXTRACTION:")
    print("="*50)
    
    # Group by source
    by_source = {}
    for skill in result.extracted_skills:
        source = skill.source
        if source not in by_source:
            by_source[source] = []
        by_source[source].append(skill)
    
    for source, skills in by_source.items():
        print(f"\n📂 {source.upper()} ({len(skills)} skills):")
        skills_sorted = sorted(skills, key=lambda x: x.confidence, reverse=True)
        for skill in skills_sorted[:10]:
            print(f"   • {skill.name} (confidence: {skill.confidence:.2f})")
            if skill.context:
                print(f"     Context: {skill.context[:100]}...")
        if len(skills) > 10:
            print(f"   ... and {len(skills) - 10} more")
    
else:
    print("❌ No analysis results available.")

## 13. Learning Recommendations

In [ ]:
# Generate learning roadmap based on skill gaps
if 'analysis_result' in locals():
    result = analysis_result
    
    print("🎓 LEARNING ROADMAP:")
    print("="*50)
    
    all_gaps = result.missing_required + result.missing_preferred
    
    if not all_gaps:
        print("🎉 Excellent! You have all the key skills for this role.")
    else:
        # Sort by priority score and learning time
        sorted_gaps = sorted(all_gaps, key=lambda x: (x.priority_score, -x.estimated_weeks), reverse=True)
        
        # Create learning phases
        phase_duration = 8  # weeks per phase
        current_phase = 1
        current_week = 0
        
        print(f"\n📅 PHASE {current_phase} (Weeks 1-{phase_duration}):")
        
        for gap in sorted_gaps:
            if current_week + gap.estimated_weeks > phase_duration:
                current_phase += 1
                current_week = 0
                print(f"\n📅 PHASE {current_phase} (Weeks {(current_phase-1)*phase_duration + 1}-{current_phase*phase_duration}):")
            
            print(f"   📚 Week {current_week + 1}-{current_week + gap.estimated_weeks}: {gap.name}")
            print(f"      Priority: {gap.priority} | Reason: {gap.reason}")
            
            if gap.has_foundation:
                print(f"      ✅ You have foundational knowledge for this skill")
            
            current_week += gap.estimated_weeks
        
        total_weeks = sum(gap.estimated_weeks for gap in sorted_gaps)
        print(f"\n⏱️  Total estimated learning time: {total_weeks} weeks ({total_weeks//4} months)")
    
else:
    print("❌ No analysis results available.")

## 14. Export Results

In [ ]:
# Export results to JSON for download
if 'analysis_result' in locals():
    result = analysis_result
    
    # Convert to JSON-serializable format
    export_data = {
        "analysis_summary": {
            "target_role": result.target_role,
            "match_score": result.match_score,
            "readiness_level": result.readiness_level,
            "confidence_score": result.confidence_score,
            "analysis_date": result.generated_at
        },
        "salary_estimate": result.salary_band_estimate,
        "skills": {
            "extracted": [
                {
                    "name": skill.name,
                    "confidence": skill.confidence,
                    "source": skill.source
                } for skill in result.extracted_skills
            ],
            "matched": result.matched_skills,
            "missing_required": [gap.dict() for gap in result.missing_required],
            "missing_preferred": [gap.dict() for gap in result.missing_preferred],
            "transferable": [ts.dict() for ts in result.transferable_skills],
            "priority_learning": result.priority_skills
        },
        "explanations": result.explanations
    }
    
    # Save to file
    import json
    filename = f"bridgr_analysis_{result.target_role.replace(' ', '_').lower()}.json"
    
    with open(filename, 'w') as f:
        json.dump(export_data, f, indent=2)
    
    # Download the file
    files.download(filename)
    
    print(f"✅ Results exported to {filename} and downloaded!")
    
else:
    print("❌ No analysis results available.")

## 15. Test with Sample Data (Optional)

In [ ]:
# Test the system with sample resume text (no PDF needed)
def test_with_sample_data():
    """Test the analysis pipeline with sample data."""
    
    sample_resume_text = """
    John Doe
    Software Engineer with 5 years of experience
    
    EXPERIENCE
    Senior Software Engineer at Tech Corp (2020-Present)
    - Developed web applications using Python, Django, and React
    - Worked with AWS for cloud deployment
    - Used Docker for containerization
    - Implemented REST APIs and microservices
    
    SKILLS
    Programming: Python, JavaScript, SQL
    Technologies: React, Django, AWS, Docker, Git
    Databases: PostgreSQL, MongoDB
    
    EDUCATION
    Bachelor of Computer Science
    University of Technology
    """
    
    # Create mock resume data
    mock_resume_data = {
        "full_text": sample_resume_text,
        "sections": {
            "experience": "Senior Software Engineer at Tech Corp (2020-Present) - Developed web applications using Python, Django, and React - Worked with AWS for cloud deployment - Used Docker for containerization - Implemented REST APIs and microservices",
            "skills": "Programming: Python, JavaScript, SQL Technologies: React, Django, AWS, Docker, Git Databases: PostgreSQL, MongoDB",
            "education": "Bachelor of Computer Science University of Technology"
        },
        "metadata": {
            "pages": 1,
            "char_count": len(sample_resume_text),
            "has_skills_section": True
        }
    }
    
    # Test skill extraction
    print("🧪 Testing skill extraction...")
    extracted_skills = analyzer.skill_extractor.extract(mock_resume_data, debug=True)
    user_skills = [s.normalized for s in extracted_skills]
    
    print(f"✅ Extracted {len(extracted_skills)} skills:")
    for skill in extracted_skills[:10]:
        print(f"   • {skill.name} (confidence: {skill.confidence:.2f})")
    
    # Test matching for different roles
    test_roles = ["Software Engineer", "Data Scientist", "Product Manager"]
    
    for role in test_roles:
        print(f"\n🎯 Testing match for {role}...")
        job_profile = get_job_skills(role)
        
        match_score, confidence = analyzer.matching_engine.compute_match(
            user_skills, job_profile["tech_skills"], job_profile["soft_skills"]
        )
        
        print(f"   Match Score: {match_score}% (confidence: {confidence})")
    
    return extracted_skills

# Run the test
print("🧪 Running sample data test...")
test_skills = test_with_sample_data()
print("\n✅ Test completed!")

# 🎉 Ready to Use!

This Colab notebook contains the complete Bridgr ML model with all components:

## ✨ Features Included:
- **Resume Parsing**: Extract text and sections from PDF resumes
- **Skill Extraction**: Multi-tier extraction using NLP and semantic similarity
- **Job Matching**: Hybrid semantic + lexical matching algorithm
- **Gap Analysis**: Prioritized skill gap identification with learning timelines
- **Transferable Skills**: Detect related skills that partially cover requirements
- **Salary Estimation**: Indian market salary bands by role
- **Learning Roadmap**: Phase-based learning recommendations

## 🚀 How to Use:
1. Run all cells in order to setup the environment
2. Upload your resume PDF when prompted
3. Enter your target job role
4. Get comprehensive analysis with actionable insights

## 📊 What You'll Get:
- Match score and readiness assessment
- Detailed skill breakdown with confidence levels
- Prioritized learning recommendations
- Transferable skill mapping
- Salary estimates for Indian market
- Exportable JSON report

## 💡 Tips:
- Use text-based PDFs (not scanned images)
- Try different target roles to compare matches
- Set your OpenAI API key for enhanced skill extraction
- Export results for future reference

The model is now ready for testing and experimentation in Google Colab! 🎯